# tJ neural-network training sweep

This notebook defines, but does not run, the requested sweep over site count, hole count, J/t, CPU/GPU backend, and Monte Carlo vs full-basis training. The expensive calls are collected in the final cell and are commented out by default.

In [1]:
using LinearAlgebra
using Random
using Serialization
using SparseArrays
using Statistics
using Dates
using Printf

try
    import Arpack
catch err
    @warn "Arpack.jl is not available; exact sparse diagonalization will fail until it is installed." exception=(err, catch_backtrace())
end

function find_tj_dir()
    candidates = unique([pwd(), joinpath(pwd(), "tJ_julia"), @__DIR__])
    for dir in candidates
        if isfile(joinpath(dir, "tJ.jl")) && isfile(joinpath(dir, "tJNetRC.jl"))
            return dir
        end
    end
    error("Could not find tJ.jl and tJNetRC.jl. Start Julia from the project root or tJ_julia/.")
end

const TJ_DIR = find_tj_dir()
include(joinpath(TJ_DIR, "tJ.jl"))
include(joinpath(TJ_DIR, "tJNetRC.jl"))

using .TJModels
using .TJNetRC

const RESULTS_DIR = joinpath(TJ_DIR, "training_sweep_results")
const MODEL_DIR = joinpath(RESULTS_DIR, "models")
mkpath(RESULTS_DIR)
mkpath(MODEL_DIR)

const TRAINING_RESULTS_PATH = joinpath(RESULTS_DIR, "training_results.tsv")
const EXACT_RESULTS_PATH = joinpath(RESULTS_DIR, "exact_ground_energies.tsv")

(TJ_DIR=TJ_DIR, RESULTS_DIR=RESULTS_DIR, MODEL_DIR=MODEL_DIR)

(TJ_DIR = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia", RESULTS_DIR = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia/training_sweep_results", MODEL_DIR = "/n/holylabs/kaxiras_lab/axzhu/projects/tJ_julia/training_sweep_results/models")

## Sweep definition

The supercell table below is editable. The 6-, 10-, and 14-site cells use compact skew cells because the skinny rectangular prime-area cells collapse nearest-neighbor bonds under periodic boundary conditions in `kneighbours(model, 1)`.

In [2]:
const SITE_COUNTS = [6, 8, 10, 12, 14, 16, 18]
const CPU_SITE_COUNTS = [6, 8, 10, 12, 14]
const HOLE_COUNTS = [0, 1, 2]
const J_RATIOS = [j / 10 for j in 0:10]
const T_VALUE = 1.0

const LATTICE_SPECS = Dict(
    6 => (m=1, n=1, mp=0, np=3),
    8 => (m=2, n=0, mp=0, np=2),
    10 => (m=2, n=1, mp=1, np=3),
    12 => (m=3, n=0, mp=0, np=2),
    14 => (m=2, n=1, mp=1, np=4),
    16 => (m=4, n=0, mp=0, np=2),
    18 => (m=3, n=0, mp=0, np=3),
)

function spin_counts(sites::Integer, holes::Integer)
    remaining = sites - holes
    remaining >= 0 || throw(ArgumentError("holes cannot exceed sites"))

    if holes == 0
        iseven(sites) || throw(ArgumentError("0-hole setting expects an even site count"))
        return (nup=sites ÷ 2, ndown=sites ÷ 2)
    elseif holes == 1
        nup = cld(remaining, 2)
        return (nup=nup, ndown=remaining - nup)
    elseif holes == 2
        iseven(remaining) || throw(ArgumentError("2-hole setting expects an even number of electrons"))
        return (nup=remaining ÷ 2, ndown=remaining ÷ 2)
    end

    throw(ArgumentError("Only 0, 1, and 2 holes are requested."))
end

function make_configurations()
    configs = NamedTuple[]
    for sites in SITE_COUNTS, holes in HOLE_COUNTS
        counts = spin_counts(sites, holes)
        spec = LATTICE_SPECS[sites]
        push!(configs, (
            sites=sites,
            holes=holes,
            nup=counts.nup,
            ndown=counts.ndown,
            lattice=spec,
        ))
    end
    return configs
end

const CONFIGURATIONS = make_configurations()
CONFIGURATIONS

21-element Vector{NamedTuple}:
 (sites = 6, holes = 0, nup = 3, ndown = 3, lattice = (m = 1, n = 1, mp = 0, np = 3))
 (sites = 6, holes = 1, nup = 3, ndown = 2, lattice = (m = 1, n = 1, mp = 0, np = 3))
 (sites = 6, holes = 2, nup = 2, ndown = 2, lattice = (m = 1, n = 1, mp = 0, np = 3))
 (sites = 8, holes = 0, nup = 4, ndown = 4, lattice = (m = 2, n = 0, mp = 0, np = 2))
 (sites = 8, holes = 1, nup = 4, ndown = 3, lattice = (m = 2, n = 0, mp = 0, np = 2))
 (sites = 8, holes = 2, nup = 3, ndown = 3, lattice = (m = 2, n = 0, mp = 0, np = 2))
 (sites = 10, holes = 0, nup = 5, ndown = 5, lattice = (m = 2, n = 1, mp = 1, np = 3))
 (sites = 10, holes = 1, nup = 5, ndown = 4, lattice = (m = 2, n = 1, mp = 1, np = 3))
 (sites = 10, holes = 2, nup = 4, ndown = 4, lattice = (m = 2, n = 1, mp = 1, np = 3))
 (sites = 12, holes = 0, nup = 6, ndown = 6, lattice = (m = 3, n = 0, mp = 0, np = 2))
 (sites = 12, holes = 1, nup = 6, ndown = 5, lattice = (m = 3, n = 0, mp = 0, np = 2))
 (sites = 12, hole

In [3]:
# Edit these defaults before launching the sweep.
const BASE_SEED = 12345

const BASE_TRAINING_PARAMS = Dict{String, Any}(
    "n_neurons" => 128,
    "epochs" => 1000,
    "dropout" => 0.0,
    "loss_diff" => 1.0e-4,
    "stagnation" => 20,
    "verbose" => true,
    "print_every" => 10,
    "model_path" => nothing,
    "load_from" => nothing,
    "exploit_symmetry" => false,
    "return_model_on_device" => false,
    "save_model_on_device" => false,
    "gpu_fallback" => false,
)

const FULL_BASIS_PARAMS = Dict{String, Any}(
    "history_size" => 200,
    "nn_learning_rate" => 1.0e-3,
)

const MC_PARAMS = Dict{String, Any}(
    "mc_optimizer" => "adam",
    "mc_learning_rate" => 1.0e-3,
    "mc_samples" => 2048,
    "mc_eval_samples" => 4096,
    "mc_chains" => 128,
    "mc_burn_in" => 64,
    "mc_eval_burn_in" => 128,
    "mc_sweeps_per_sample" => 1,
    "return_wavefunction" => false,
)

const EXACT_DIAG_K = 1
const EXACT_DIAG_TOL = 1.0e-9
const EXACT_DIAG_MAXITER = 3000

(BASE_TRAINING_PARAMS=BASE_TRAINING_PARAMS, FULL_BASIS_PARAMS=FULL_BASIS_PARAMS, MC_PARAMS=MC_PARAMS)

(BASE_TRAINING_PARAMS = Dict{String, Any}("n_neurons" => 128, "verbose" => true, "print_every" => 10, "epochs" => 1000, "model_path" => nothing, "gpu_fallback" => false, "exploit_symmetry" => false, "loss_diff" => 0.0001, "dropout" => 0.0, "stagnation" => 20…), FULL_BASIS_PARAMS = Dict{String, Any}("history_size" => 200, "nn_learning_rate" => 0.001), MC_PARAMS = Dict{String, Any}("return_wavefunction" => false, "mc_burn_in" => 64, "mc_optimizer" => "adam", "mc_samples" => 2048, "mc_eval_burn_in" => 128, "mc_learning_rate" => 0.001, "mc_eval_samples" => 4096, "mc_chains" => 128, "mc_sweeps_per_sample" => 1))

In [4]:
const TRAINING_COLUMNS = [
    "timestamp", "status", "error", "elapsed_seconds",
    "sites", "holes", "nup", "ndown", "dim",
    "lattice_m", "lattice_n", "lattice_mp", "lattice_np",
    "backend", "use_mc", "t", "J_over_t", "tcoef", "Jcoef",
    "energy", "energy_variance", "energy_stderr",
    "nn_spin_corr", "nn_spin_corr_variance", "nn_spin_corr_stderr",
    "acceptance", "accepted", "proposed", "seed", "model_path",
]

const EXACT_COLUMNS = [
    "timestamp", "status", "error", "elapsed_seconds",
    "sites", "holes", "nup", "ndown", "dim",
    "lattice_m", "lattice_n", "lattice_mp", "lattice_np",
    "t", "J_over_t", "tcoef", "Jcoef",
    "exact_ground_energy"
]

function tsv_cell(x)
    (x === missing || x === nothing) && return ""
    s = string(x)
    return replace(s, '\t' => ' ', '\n' => ' ')
end

function append_tsv_row!(path::AbstractString, columns::Vector{String}, row::AbstractDict)
    new_file = !isfile(path) || filesize(path) == 0
    open(path, "a") do io
        new_file && println(io, join(columns, "\t"))
        println(io, join((tsv_cell(get(row, col, missing)) for col in columns), "\t"))
    end
    return path
end

function base_row(cfg, problem, J_ratio)
    spec = cfg.lattice
    tcoef = -T_VALUE
    Jcoef = J_ratio * T_VALUE
    return Dict{String, Any}(
        "timestamp" => Dates.format(now(), dateformat"yyyy-mm-ddTHH:MM:SS"),
        "sites" => cfg.sites,
        "holes" => cfg.holes,
        "nup" => cfg.nup,
        "ndown" => cfg.ndown,
        "dim" => problem.model.dim,
        "lattice_m" => spec.m,
        "lattice_n" => spec.n,
        "lattice_mp" => spec.mp,
        "lattice_np" => spec.np,
        "t" => T_VALUE,
        "J_over_t" => J_ratio,
        "tcoef" => tcoef,
        "Jcoef" => Jcoef,
    )
end

base_row (generic function with 1 method)

In [5]:
function config_id(cfg)
    return @sprintf("M%02d_h%d_up%02d_dn%02d", cfg.sites, cfg.holes, cfg.nup, cfg.ndown)
end

j_id(J_ratio) = @sprintf("J%02d", round(Int, 10 * J_ratio))

function model_path_for(cfg, backend::AbstractString, use_mc::Bool, J_ratio, seed::Integer)
    mc_label = use_mc ? "mc" : "full"
    filename = "$(config_id(cfg))_$(backend)_$(mc_label)_$(j_id(J_ratio))_seed$(seed).jls"
    return joinpath(MODEL_DIR, filename)
end

function nn_spin_corr_value(neighbours, state)
    total = 0.0
    for (i, j) in neighbours
        total += 0.25 * state[i] * state[j]
    end
    return total / length(neighbours)
end

function nn_spin_corr_diag(model, neighbours)
    vals = Vector{Float64}(undef, model.dim)
    for (idx, state) in enumerate(model.basis)
        vals[idx] = nn_spin_corr_value(neighbours, state)
    end
    return vals
end

function weighted_mean(values, weights)
    norm_sq = sum(weights)
    norm_sq == 0 && return NaN
    return sum(values .* weights) / norm_sq
end

function psi_nn_spin_corr(psi, spin_diag)
    weights = abs2.(psi)
    return weighted_mean(spin_diag, weights)
end

function mc_nn_spin_corr_stats(neighbours, samples)
    values = [nn_spin_corr_value(neighbours, state) for state in samples]
    mean_value = mean(values)
    variance = length(values) > 1 ? var(values; corrected=true) : 0.0
    stderr = sqrt(variance / length(values))
    return (mean=mean_value, variance=variance, stderr=stderr, values=values)
end

function prepare_problem(cfg)
    spec = cfg.lattice
    model = TJModels.TJ(cfg.nup, cfg.ndown, spec.m, spec.n, spec.mp, spec.np)
    neighbours = TJModels.kneighbours(model, 1)
    expected_neighbours = 3 * model.M ÷ 2
    length(neighbours) == expected_neighbours || @warn "Unexpected nearest-neighbor count" sites=cfg.sites neighbours=length(neighbours) expected=expected_neighbours

    terms = TJModels.hamiltonian_by_degree(model; degrees=(1,))
    mats = TJModels.assemble_by_degree(model, terms)
    Ht = mats.t[1]
    HJ = mats.J[1]
    spin_diag = nn_spin_corr_diag(model, neighbours)

    return (model=model, Ht=Ht, HJ=HJ, neighbours=neighbours, spin_diag=spin_diag)
end

function hamiltonian(problem, J_ratio)
    tcoef = -T_VALUE
    Jcoef = J_ratio * T_VALUE
    return tcoef .* problem.Ht .+ Jcoef .* problem.HJ
end

hamiltonian (generic function with 1 method)

In [6]:
function diagonalize_ground_state(H; k=EXACT_DIAG_K, tol=EXACT_DIAG_TOL, maxiter=EXACT_DIAG_MAXITER)
    if issparse(H)
        isdefined(Main, :Arpack) || throw(ArgumentError("Sparse diagonalization requires Arpack.jl."))
        nev = min(k, size(H, 1) - 2)
        nev > 0 || throw(ArgumentError("Sparse diagonalization needs matrix size at least 3."))
        evals, evecs, nconv, niter, nmult, resid = Arpack.eigs(
            H;
            nev=nev,
            which=:SR,
            ritzvec=true,
            tol=tol,
            maxiter=maxiter,
        )
        order = sortperm(real.(evals))
        return (
            ground_energy=Float64(real(evals[order[1]])),
            nconv=nconv,
            niter=niter,
            nmult=nmult,
            residual=resid,
        )
    end

    F = eigen(Hermitian(Matrix(H)))
    return (
        ground_energy=Float64(minimum(F.values)),
        nconv=missing,
        niter=missing,
        nmult=missing,
        residual=missing,
    )
end

function run_exact_one!(cfg, problem, J_ratio)
    row = base_row(cfg, problem, J_ratio)
    elapsed = @elapsed begin
        try
            spectrum = diagonalize_ground_state(hamiltonian(problem, J_ratio))
            merge!(row, Dict{String, Any}(
                "status" => "ok",
                "error" => "",
                "exact_ground_energy" => spectrum.ground_energy,
            ))
        catch err
            merge!(row, Dict{String, Any}(
                "status" => "error",
                "error" => sprint(showerror, err),
                "exact_ground_energy" => missing,
            ))
        end
    end
    row["elapsed_seconds"] = elapsed
    append_tsv_row!(EXACT_RESULTS_PATH, EXACT_COLUMNS, row)
    return row
end

run_exact_one! (generic function with 1 method)

In [7]:
function merged_params(dicts::AbstractDict...)
    out = Dict{String, Any}()
    for dict in dicts
        for (key, value) in dict
            out[string(key)] = value
        end
    end
    return out
end

function seed_for_task(cfg, J_ratio, backend::AbstractString, use_mc::Bool)
    j_index = round(Int, 10 * J_ratio)
    backend_offset = backend == "gpu" ? 10 : 20
    mc_offset = use_mc ? 1 : 0
    return BASE_SEED + 100000 * cfg.sites + 10000 * cfg.holes + 100 * j_index + backend_offset + mc_offset
end

function params_for_task(cfg, J_ratio, backend::AbstractString, use_mc::Bool, seed::Integer)
    params = use_mc ? merged_params(BASE_TRAINING_PARAMS, MC_PARAMS) : merged_params(BASE_TRAINING_PARAMS, FULL_BASIS_PARAMS)
    params["device"] = backend
    params["model_path"] = model_path_for(cfg, backend, use_mc, J_ratio, seed)
    params["load_from"] = nothing
    params["use_monte_carlo"] = use_mc

    if use_mc
        params["mc_batched"] = backend == "gpu"
        params["return_wavefunction"] = false
    else
        params["nn_optimizer"] = backend == "gpu" ? "adam" : "lbfgs"
        params["return_wavefunction"] = true
    end

    return params
end

function run_training_one!(cfg, problem, J_ratio, backend::AbstractString, use_mc::Bool)
    seed = seed_for_task(cfg, J_ratio, backend, use_mc)
    params = params_for_task(cfg, J_ratio, backend, use_mc, seed)
    row = base_row(cfg, problem, J_ratio)
    merge!(row, Dict{String, Any}(
        "backend" => backend,
        "use_mc" => use_mc,
        "seed" => seed,
        "model_path" => params["model_path"],
    ))

    elapsed = @elapsed begin
        try
            if use_mc
                result = TJNetRC.trainNNMC(
                    params,
                    problem.model,
                    row["tcoef"],
                    row["Jcoef"],
                    problem.Ht,
                    problem.HJ;
                    seed=seed,
                    return_details=true,
                )
                corr_stats = mc_nn_spin_corr_stats(problem.neighbours, result.final_stats.samples)
                merge!(row, Dict{String, Any}(
                    "status" => "ok",
                    "error" => "",
                    "energy" => result.energy,
                    "energy_variance" => result.variance,
                    "energy_stderr" => result.stderr,
                    "nn_spin_corr" => corr_stats.mean,
                    "nn_spin_corr_variance" => corr_stats.variance,
                    "nn_spin_corr_stderr" => corr_stats.stderr,
                    "acceptance" => result.final_stats.acceptance,
                    "accepted" => result.final_stats.accepted,
                    "proposed" => result.final_stats.proposed,
                ))
            else
                psi, energy = TJNetRC.trainNN(
                    params,
                    problem.model,
                    row["tcoef"],
                    row["Jcoef"],
                    problem.Ht,
                    problem.HJ;
                    seed=seed,
                )
                merge!(row, Dict{String, Any}(
                    "status" => "ok",
                    "error" => "",
                    "energy" => energy,
                    "energy_variance" => missing,
                    "energy_stderr" => missing,
                    "nn_spin_corr" => psi_nn_spin_corr(psi, problem.spin_diag),
                    "nn_spin_corr_variance" => missing,
                    "nn_spin_corr_stderr" => missing,
                    "acceptance" => missing,
                    "accepted" => missing,
                    "proposed" => missing,
                ))
            end
        catch err
            merge!(row, Dict{String, Any}(
                "status" => "error",
                "error" => sprint(showerror, err),
                "energy" => missing,
                "energy_variance" => missing,
                "energy_stderr" => missing,
                "nn_spin_corr" => missing,
                "nn_spin_corr_variance" => missing,
                "nn_spin_corr_stderr" => missing,
                "acceptance" => missing,
                "accepted" => missing,
                "proposed" => missing,
            ))
        end
    end

    row["elapsed_seconds"] = elapsed
    append_tsv_row!(TRAINING_RESULTS_PATH, TRAINING_COLUMNS, row)
    GC.gc()
    return row
end

run_training_one! (generic function with 1 method)

In [8]:
function backends_for_sites(sites::Integer)
    return sites in CPU_SITE_COUNTS ? ["gpu", "cpu"] : ["gpu"]
end

function run_exact_sweep!()
    rows = Dict{String, Any}[]
    for cfg in CONFIGURATIONS
        @info "Preparing exact problem" sites=cfg.sites holes=cfg.holes nup=cfg.nup ndown=cfg.ndown
        problem = prepare_problem(cfg)
        for J_ratio in J_RATIOS
            @info "Diagonalizing" sites=cfg.sites holes=cfg.holes J_over_t=J_ratio dim=problem.model.dim
            push!(rows, run_exact_one!(cfg, problem, J_ratio))
        end
        GC.gc()
    end
    return rows
end

function run_training_sweep!(; requested_backends=nothing, requested_mc=nothing)
    rows = Dict{String, Any}[]
    backend_filter = requested_backends === nothing ? nothing : Set(String.(requested_backends))
    mc_filter = requested_mc === nothing ? nothing : Set(Bool.(requested_mc))

    for cfg in CONFIGURATIONS
        @info "Preparing training problem" sites=cfg.sites holes=cfg.holes nup=cfg.nup ndown=cfg.ndown
        problem = prepare_problem(cfg)

        for J_ratio in J_RATIOS
            for backend in backends_for_sites(cfg.sites)
                backend_filter === nothing || backend in backend_filter || continue
                for use_mc in (false, true)
                    mc_filter === nothing || use_mc in mc_filter || continue
                    @info "Training" sites=cfg.sites holes=cfg.holes J_over_t=J_ratio backend=backend use_mc=use_mc dim=problem.model.dim
                    push!(rows, run_training_one!(cfg, problem, J_ratio, backend, use_mc))
                end
            end
        end
        GC.gc()
    end
    return rows
end

function sweep_counts()
    exact_count = length(CONFIGURATIONS) * length(J_RATIOS)
    training_count = 0
    for cfg in CONFIGURATIONS
        training_count += length(J_RATIOS) * length(backends_for_sites(cfg.sites)) * 2
    end
    return (exact=exact_count, training=training_count, total=exact_count + training_count)
end

sweep_counts()

(exact = 231, training = 792, total = 1023)

## Launch cells

The calls below are intentionally commented out. Uncomment only the subset you want to run in a given job allocation.

In [9]:
# Exact diagonalization for all 21 site/hole/spin settings and all 11 J/t values.
exact_rows = run_exact_sweep!()

# Useful subsets for separate cluster jobs:
# gpu_full_rows = run_training_sweep!(requested_backends=["gpu"], requested_mc=[false])
# gpu_mc_rows = run_training_sweep!(requested_backends=["gpu"], requested_mc=[true])
# cpu_full_rows = run_training_sweep!(requested_backends=["cpu"], requested_mc=[false])

┌ Info: Preparing exact problem
│   sites = 6
│   holes = 0
│   nup = 3
└   ndown = 3
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.0
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.1
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.2
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.3
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.4
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.5
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.6
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.7
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.8
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 0.9
└   dim = 20
┌ Info: Diagonalizing
│   sites = 6
│   holes = 0
│   J_over_t = 1.0
└   dim = 20
┌ Info: Prep

231-element Vector{Dict{String, Any}}:
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.0, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" => "ok", "sites" => 6…)
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.1, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" => "ok", "sites" => 6…)
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.2, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" => "ok", "sites" => 6…)
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.3, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" => "ok", "sites" => 6…)
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.4, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" => "ok", "sites" => 6…)
 Dict("lattice_m" => 1, "tcoef" => -1.0, "ndown" => 3, "error" => "", "Jcoef" => 0.5, "nup" => 3, "holes" => 0, "lattice_np" => 3, "status" =